In [1]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

In [2]:
import { RecursiveSet } from './Recursive-Set';

type Variable = string;
type Literal = Variable | ['¬', Variable];
type Clause = RecursiveSet<Literal>;

# Sudoku

In [3]:
import * as DP from './06-Davis-Putnam';

The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is defined below.

In [4]:
function createPuzzle(): (number | string)[][] {
    return [
        [ 8 , '*', '*', '*', '*', '*', '*', '*', '*'],
        ['*', '*',  3,   6 , '*', '*', '*', '*', '*'],
        ['*',  7 , '*', '*',  9 , '*',  2 , '*', '*'],
        ['*',  5 , '*', '*', '*',  7 , '*', '*', '*'],
        ['*', '*', '*', '*',  4 ,  5 ,  7 , '*', '*'],
        ['*', '*', '*',  1 , '*', '*', '*',  3 , '*'],
        ['*', '*',  1 , '*', '*', '*', '*',  6 ,  8 ],
        ['*', '*',  8 ,  5 , '*', '*', '*',  1 , '*'],
        ['*',  9 , '*', '*', '*', '*',  4 , '*', '*']
    ];
}

We will solve this Sudoku using the Davis-Putnam algorithm.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `varName(row, col, digit)` returns a formated string that is interpreted as a variable name.

In [5]:
function varName(row: number, col: number, digit: number): string {
    return `Q<${row},${col},${digit}>`;
}

In [6]:
varName(1,2,3);

Q<1,2,3>


The function `atMostOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at most one of the variables of `S` is true.

In [7]:
function atMostOne(S: RecursiveSet<Variable>): RecursiveSet<Clause> {
    const result = new RecursiveSet<Clause>();
    for (const p of S) {
        for (const q of S) {
            if (p < q) {
                const lit1: Literal = ['¬', p];
                const lit2: Literal = ['¬', q];
                result.add(new RecursiveSet<Literal>(lit1, lit2));
            }
        }
    }
    return result;
}

The function `atLeastOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at least one of the variables of `S` is true.

In [8]:
function atLeastOne(S: RecursiveSet<Variable>): RecursiveSet<Clause> {
    return new RecursiveSet<Clause>(new RecursiveSet<Literal>(...S));
}

The function `exactlyOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that exactly one of the variables of `S` is true.

In [9]:
function exactlyOne(S: RecursiveSet<Variable>): RecursiveSet<Clause> {
    const atMost = atMostOne(S);
    const atLeast = atLeastOne(S);
    return atMost.union(atLeast);
}

In [10]:
exactlyOne(new RecursiveSet<Variable>('a', 'b', 'c'));

{{['¬', 'a'], ['¬', 'c']}, {['¬', 'a'], ['¬', 'b']}, {a, b, c}, {['¬', 'b'], ['¬', 'c']}}


The function `exactlyOnce` takes an array `L` of pairs of indices as its argument.  The elements of `L` are pairs of the form
`(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
It returns a set of formulas expressing that all Sudoku fields specified by the coordinate pairs in `L` take different digits as values.

In [11]:
function exactlyOnce(L: Array<[number, number]>): RecursiveSet<Clause> {
    let Clauses = new RecursiveSet<Clause>();
    for (let digit = 1; digit <= 9; digit++) {
        const varNames = L.map(([row, col]) => varName(row, col, digit));
        const vars = new RecursiveSet<Variable>(...varNames);
        const exact = exactlyOne(vars);
        for (const clause of exact) {
            Clauses.add(clause);
        }
    }
    return Clauses;
}

In [12]:
const result = exactlyOnce(
    Array.from({ length: 9 }, (_, i) => [1, i + 1] as [number, number])
);
for (const clause of result) {
    console.log(`{${[...clause].map(lit => JSON.stringify(lit)).join(', ')}}`);
}

{["¬","Q<1,2,2>"], ["¬","Q<1,4,2>"]}
{["¬","Q<1,3,9>"], ["¬","Q<1,5,9>"]}
{["¬","Q<1,7,4>"], ["¬","Q<1,8,4>"]}
{["¬","Q<1,2,6>"], ["¬","Q<1,4,6>"]}
{["¬","Q<1,7,7>"], ["¬","Q<1,9,7>"]}
{["¬","Q<1,7,3>"], ["¬","Q<1,9,3>"]}
{["¬","Q<1,1,8>"], ["¬","Q<1,3,8>"]}
{["¬","Q<1,1,2>"], ["¬","Q<1,3,2>"]}
{["¬","Q<1,1,6>"], ["¬","Q<1,3,6>"]}
{["¬","Q<1,5,9>"], ["¬","Q<1,7,9>"]}
{["¬","Q<1,5,7>"], ["¬","Q<1,7,7>"]}
{["¬","Q<1,6,7>"], ["¬","Q<1,8,7>"]}
{["¬","Q<1,5,3>"], ["¬","Q<1,7,3>"]}
{["¬","Q<1,6,3>"], ["¬","Q<1,8,3>"]}
{["¬","Q<1,6,1>"], ["¬","Q<1,8,1>"]}
{["¬","Q<1,4,1>"], ["¬","Q<1,6,1>"]}
{["¬","Q<1,4,3>"], ["¬","Q<1,6,3>"]}
{["¬","Q<1,3,2>"], ["¬","Q<1,7,2>"]}
{["¬","Q<1,6,9>"], ["¬","Q<1,8,9>"]}
{["¬","Q<1,3,6>"], ["¬","Q<1,7,6>"]}
{["¬","Q<1,4,9>"], ["¬","Q<1,6,9>"]}
{["¬","Q<1,3,4>"], ["¬","Q<1,7,4>"]}
{["¬","Q<1,3,8>"], ["¬","Q<1,7,8>"]}
{["¬","Q<1,8,3>"], ["¬","Q<1,9,3>"]}
{["¬","Q<1,6,8>"], ["¬","Q<1,9,8>"]}
{["¬","Q<1,2,3>"], ["¬","Q<1,7,3>"]}
{["¬","Q<1,8,7>"], ["¬","Q<1,9,7>"]}
{

{["¬","Q<1,1,4>"], ["¬","Q<1,4,4>"]}
{["¬","Q<1,3,5>"], ["¬","Q<1,9,5>"]}
{["¬","Q<1,3,9>"], ["¬","Q<1,9,9>"]}
{["¬","Q<1,5,9>"], ["¬","Q<1,6,9>"]}
{["¬","Q<1,6,6>"], ["¬","Q<1,7,6>"]}
{["¬","Q<1,5,5>"], ["¬","Q<1,6,5>"]}
{["¬","Q<1,5,7>"], ["¬","Q<1,6,7>"]}
{["¬","Q<1,6,4>"], ["¬","Q<1,7,4>"]}
{["¬","Q<1,6,2>"], ["¬","Q<1,7,2>"]}
{["¬","Q<1,5,3>"], ["¬","Q<1,6,3>"]}
{["¬","Q<1,6,8>"], ["¬","Q<1,7,8>"]}
{"Q<1,1,3>", "Q<1,2,3>", "Q<1,3,3>", "Q<1,4,3>", "Q<1,5,3>", "Q<1,6,3>", "Q<1,7,3>", "Q<1,8,3>", "Q<1,9,3>"}
{["¬","Q<1,7,7>"], ["¬","Q<1,8,7>"]}
{["¬","Q<1,4,4>"], ["¬","Q<1,8,4>"]}
{["¬","Q<1,2,3>"], ["¬","Q<1,6,3>"]}
{["¬","Q<1,3,2>"], ["¬","Q<1,6,2>"]}
{["¬","Q<1,7,5>"], ["¬","Q<1,8,5>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,6,1>"]}
{["¬","Q<1,7,3>"], ["¬","Q<1,8,3>"]}
{["¬","Q<1,3,2>"], ["¬","Q<1,5,2>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,6,7>"]}
{["¬","Q<1,3,6>"], ["¬","Q<1,6,6>"]}
{["¬","Q<1,7,1>"], ["¬","Q<1,8,1>"]}
{["¬","Q<1,3,4>"], ["¬","Q<1,6,4>"]}
{["¬","Q<1,3,6>"], ["¬","Q<1,5,6>"]}
{["

The function `exactlyOneDigit(row, col)` takes integers `row` and `col` as arguments.  These specify the row and column of a field in a Sudoku.  The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `true`.

In [13]:
function exactlyOneDigit(row: number, col: number): RecursiveSet<Clause> {
    const vars = new RecursiveSet<Variable>();  
    for (let digit = 1; digit <= 9; digit++) {
        vars.add(varName(row, col, digit));
    }
    return exactlyOne(vars);
}

In [14]:
for (const clause of exactlyOneDigit(1, 1)) {
    console.log(clause.toString()); 
}

{['¬', 'Q<1,1,1>'], ['¬', 'Q<1,1,9>']}
{['¬', 'Q<1,1,1>'], ['¬', 'Q<1,1,5>']}
{['¬', 'Q<1,1,1>'], ['¬', 'Q<1,1,7>']}
{['¬', 'Q<1,1,3>'], ['¬', 'Q<1,1,9>']}
{['¬', 'Q<1,1,3>'], ['¬', 'Q<1,1,5>']}
{['¬', 'Q<1,1,3>'], ['¬', 'Q<1,1,7>']}
{['¬', 'Q<1,1,8>'], ['¬', 'Q<1,1,9>']}
{['¬', 'Q<1,1,4>'], ['¬', 'Q<1,1,8>']}
{['¬', 'Q<1,1,4>'], ['¬', 'Q<1,1,6>']}
{['¬', 'Q<1,1,6>'], ['¬', 'Q<1,1,8>']}
{['¬', 'Q<1,1,2>'], ['¬', 'Q<1,1,3>']}
{['¬', 'Q<1,1,4>'], ['¬', 'Q<1,1,9>']}
{['¬', 'Q<1,1,4>'], ['¬', 'Q<1,1,5>']}
{['¬', 'Q<1,1,4>'], ['¬', 'Q<1,1,7>']}
{['¬', 'Q<1,1,6>'], ['¬', 'Q<1,1,9>']}
{['¬', 'Q<1,1,6>'], ['¬', 'Q<1,1,7>']}
{['¬', 'Q<1,1,1>'], ['¬', 'Q<1,1,2>']}
{['¬', 'Q<1,1,5>'], ['¬', 'Q<1,1,8>']}
{['¬', 'Q<1,1,5>'], ['¬', 'Q<1,1,6>']}
{['¬', 'Q<1,1,7>'], ['¬', 'Q<1,1,8>']}
{['¬', 'Q<1,1,1>'], ['¬', 'Q<1,1,3>']}
{['¬', 'Q<1,1,5>'], ['¬', 'Q<1,1,9>']}
{['¬', 'Q<1,1,5>'], ['¬', 'Q<1,1,7>']}
{['¬', 'Q<1,1,7>'], ['¬', 'Q<1,1,9>']}
{Q<1,1,1>, Q<1,1,2>, Q<1,1,3>, Q<1,1,4>, Q<1,1,5>, Q<1,1,6>, Q<1

The function `constraintsFromPuzzle`  returns a set of clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.

In [15]:
function constraintsFromPuzzle(): RecursiveSet<Clause> {
    const Puzzle = createPuzzle();
    const Clauses = new RecursiveSet<Clause>();
    for (let row = 0; row < 9; row++) {
        for (let col = 0; col < 9; col++) {
            const value = Puzzle[row][col];
            if (value !== '*') {
                const v = varName(row + 1, col + 1, value as number);
                const unitClause = new RecursiveSet<Literal>(v);
                Clauses.add(unitClause);
            }
        }
    }
    return Clauses;
}

In [16]:
for (const clause of constraintsFromPuzzle()) {
    console.log(clause.toString()); 
}
constraintsFromPuzzle().size

{Q<4,2,5>}
{Q<2,3,3>}
{Q<3,5,9>}
{Q<8,8,1>}
{Q<2,4,6>}
{Q<5,7,7>}
{Q<3,7,2>}
{Q<1,1,8>}
{Q<9,2,9>}
{Q<5,5,4>}
{Q<4,6,7>}
{Q<9,7,4>}
{Q<6,4,1>}
{Q<3,2,7>}
{Q<7,3,1>}
{Q<8,4,5>}
{Q<7,8,6>}
{Q<8,3,8>}
{Q<6,8,3>}
{Q<5,6,5>}
{Q<7,9,8>}
21


The function `allConstraints` returns a CSP that encodes the given sudoku as a CSP.

In [17]:
function allConstraints(): RecursiveSet<Clause> {
    const L = [1, 2, 3, 4, 5, 6, 7, 8, 9];
    // 1. Start with constraints from the puzzle
    let Clauses = constraintsFromPuzzle();
    // 2. There is exactly one digit in every field
    for (const row of L) {
        for (const col of L) {
            const digitConstraints = exactlyOneDigit(row, col);
            // Union nutzen ist effizienter als einzelne adds in Schleife
            // Hinweis: union gibt ein NEUES Set zurück, also zuweisen!
            Clauses = Clauses.union(digitConstraints);
        }
    }
    // 3. All entries in a row are unique
    for (const row of L) {
        const rowCells = L.map(col => [row, col] as [number, number]);
        Clauses = Clauses.union(exactlyOnce(rowCells));
    }
    // 4. All entries in a column are unique
    for (const col of L) {
        const colCells = L.map(row => [row, col] as [number, number]);
        Clauses = Clauses.union(exactlyOnce(colCells));
    }
    // 5. All entries in a 3x3 square are unique
    for (let r = 0; r < 3; r++) {
        for (let c = 0; c < 3; c++) {
            const blockCells: Array<[number, number]> = [];
            for (let row = 1; row <= 3; row++) {
                for (let col = 1; col <= 3; col++) {
                    blockCells.push([r * 3 + row, c * 3 + col]);
                }
            }
            Clauses = Clauses.union(exactlyOnce(blockCells));
        }
    }
    return Clauses;
}

In [18]:
const clauses = allConstraints();
console.log("--- Clauses with size 1 ---");
for (const clause of clauses) {
    // Zugriff auf size Eigenschaft
    if (clause.size === 1) {
        console.log(clause.toString());
    }
}


--- Clauses with size 1 ---
{Q<4,2,5>}
{Q<2,3,3>}
{Q<3,5,9>}
{Q<8,8,1>}
{Q<2,4,6>}
{Q<5,7,7>}
{Q<3,7,2>}
{Q<1,1,8>}
{Q<9,2,9>}
{Q<5,5,4>}
{Q<4,6,7>}
{Q<9,7,4>}
{Q<6,4,1>}
{Q<3,2,7>}
{Q<7,3,1>}
{Q<8,4,5>}
{Q<7,8,6>}
{Q<8,3,8>}
{Q<6,8,3>}
{Q<5,6,5>}
{Q<7,9,8>}


In [19]:
console.log("\n--- Clauses with size 9 ---");
for (const clause of clauses) {
    if (clause.size === 9) {
        console.log(clause.toString());
    }
}


--- Clauses with size 9 ---
{Q<9,1,5>, Q<9,2,5>, Q<9,3,5>, Q<9,4,5>, Q<9,5,5>, Q<9,6,5>, Q<9,7,5>, Q<9,8,5>, Q<9,9,5>}
{Q<7,7,4>, Q<7,8,4>, Q<7,9,4>, Q<8,7,4>, Q<8,8,4>, Q<8,9,4>, Q<9,7,4>, Q<9,8,4>, Q<9,9,4>}
{Q<4,1,7>, Q<4,2,7>, Q<4,3,7>, Q<5,1,7>, Q<5,2,7>, Q<5,3,7>, Q<6,1,7>, Q<6,2,7>, Q<6,3,7>}
{Q<8,7,1>, Q<8,7,2>, Q<8,7,3>, Q<8,7,4>, Q<8,7,5>, Q<8,7,6>, Q<8,7,7>, Q<8,7,8>, Q<8,7,9>}
{Q<1,3,2>, Q<2,3,2>, Q<3,3,2>, Q<4,3,2>, Q<5,3,2>, Q<6,3,2>, Q<7,3,2>, Q<8,3,2>, Q<9,3,2>}
{Q<7,7,8>, Q<7,8,8>, Q<7,9,8>, Q<8,7,8>, Q<8,8,8>, Q<8,9,8>, Q<9,7,8>, Q<9,8,8>, Q<9,9,8>}
{Q<9,3,1>, Q<9,3,2>, Q<9,3,3>, Q<9,3,4>, Q<9,3,5>, Q<9,3,6>, Q<9,3,7>, Q<9,3,8>, Q<9,3,9>}
{Q<8,3,1>, Q<8,3,2>, Q<8,3,3>, Q<8,3,4>, Q<8,3,5>, Q<8,3,6>, Q<8,3,7>, Q<8,3,8>, Q<8,3,9>}
{Q<6,1,6>, Q<6,2,6>, Q<6,3,6>, Q<6,4,6>, Q<6,5,6>, Q<6,6,6>, Q<6,7,6>, Q<6,8,6>, Q<6,9,6>}
{Q<1,7,1>, Q<2,7,1>, Q<3,7,1>, Q<4,7,1>, Q<5,7,1>, Q<6,7,1>, Q<7,7,1>, Q<8,7,1>, Q<9,7,1>}
{Q<4,2,1>, Q<4,2,2>, Q<4,2,3>, Q<4,2,4>, Q<4,2,5>, Q<4,2,6>, 

In [20]:
clauses.size

10551


The function `solve(Constraints, Variables)` receives two arguments:
- `Constraints` is a set of formulas representing a constraint satisfaction problem.
- `Variables`   is the set of variables that occur in this formulas.

The function computes a solution to the given problem and returns this solution.

In [21]:
function sudoku(): RecursiveSet<Clause> | null {
    const Clauses = allConstraints();
    const Solution = DP.solve(Clauses);
    const EmptyClause = new RecursiveSet<Literal>();
    if (!Solution.has(EmptyClause)) {
        return Solution;
    } else {
        console.log('The problem is not solvable!');
        return null;
    }
}

In [22]:
console.time('sudoku');
const Solution = sudoku();
console.timeEnd('sudoku');

sudoku: 1:05:36.089 (h:mm:ss.mmm)


## Graphical Representation

In [1]:
function transformSolution(Solution: RecursiveSet<Clause>): Record<string, number> {
    const Result: Record<string, number> = {};
    for (const UnitClause of Solution) {
        const literal = DP.arb(UnitClause) as Literal;
        if (typeof literal === 'string') {
            const matches = literal.match(/(\d+),(\d+),(\d+)/);
            
            if (matches) {
                const row = parseInt(matches[1]);
                const col = parseInt(matches[2]);
                const digit = parseInt(matches[3]);
                Result[`V${row}${col}`] = digit;
            }
        }
    }
    return Result;
}

1:38 - Cannot find name 'RecursiveSet'.
1:38 - Parameter 'Solution' of exported function has or is using private name 'RecursiveSet'.
1:51 - Cannot find name 'Clause'.
1:51 - Parameter 'Solution' of exported function has or is using private name 'Clause'.
4:25 - Cannot find name 'DP'.
4:47 - Cannot find name 'Literal'.


In [24]:
import { display } from 'tslab';
function showSolution(Solution: RecursiveSet<Clause>, width: string = '50%'): void {
    const solutionMap: Record<string, number> = transformSolution(Solution);
    const Sudoku = createPuzzle();
    for (let row = 0; row < 9; row++) {
        for (let col = 0; col < 9; col++) {
            if (Sudoku[row][col] !== '*') {
                delete solutionMap[`V${row + 1}${col + 1}`];
            }
        }
    }
    let html = `<table style="width:${width}; border-collapse: collapse; border: 2px solid black; font-family: sans-serif;">`;
    for (let row = 0; row < 9; row++) {
        html += '<tr>';
        for (let col = 0; col < 9; col++) {
            const key = `V${row + 1}${col + 1}`;
            let value = solutionMap[key];
            const original = Sudoku[row][col];
            let cellStyle = "font-weight: normal;";
            if (original !== '*') {
                value = original as number;
                cellStyle = "font-weight: bold;";
            }
            const blockRow = Math.floor(row / 3);
            const blockCol = Math.floor(col / 3);
            const isGray = (blockRow + blockCol) % 2 !== 0;
            const bgColor = isGray ? '#f0f0f0' : '#ffffff';
            let borderStyle = "border: 1px solid #ccc;";
            if ((col + 1) % 3 === 0 && col < 8) borderStyle += "border-right: 2px solid black;";
            if ((row + 1) % 3 === 0 && row < 8) borderStyle += "border-bottom: 2px solid black;";
            html += `<td style="${borderStyle} width:30px; height:30px; text-align:center; font-size:20px; background-color:${bgColor}; ${cellStyle}">${value || ''}</td>`;
        }
        html += '</tr>';
    }
    html += '</table>';
    display.html(html);
}

In [ ]:
showSolution(Solution);

unexpected error: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at updateContent (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:601:9)
    at Object.convert (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:252:9)
    at Object.execute (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\executor.js:138:38)
    at JupyterHandlerImp